# Simulation 2 - CRT - WGAN - Figures
This notebook takes outputs from the Simulation 2 WGAN experiment and regenerates the cleaned manuscript figures. To run, change the input paths to the output of the desired run. 

Dependencies: json, numpy, matplotlib, scipy

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from matplotlib.ticker import FormatStrFormatter

# ============================================================
# Output Formats
# ============================================================
SAVE_PDF = True
SAVE_PNG = True   # Generates both PNG and PDF if True

# ============================================================
# Update sizing (Publication-Quality Style)
# ============================================================
import matplotlib as mpl 
mpl.rcParams["font.sans-serif"] = ["Arial"]
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.titlesize"] = 26
mpl.rcParams["axes.labelsize"] = 26
mpl.rcParams["legend.fontsize"] = 17 # was 15
mpl.rcParams["legend.title_fontsize"] = 16
mpl.rcParams["xtick.labelsize"] = 24
mpl.rcParams["ytick.labelsize"] = 24

# Cleaner editable text in PDF/SVG
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

# ============================================================
# Point this to the saved NN run folder
# ============================================================
# Update this path to target the specific timestamped directory
RUN_DIR = Path("logs/NNW1/FINAL-NNW1_sorted_mlp_alphaReal_Tvar_mTotal=500_nreps=100_MMDsample_KDEplot_20260531-124347")
RESULTS_PATH = RUN_DIR / "all_results.json"

REMAKE_FIG_DIR = RUN_DIR / "remade_figures"
REMAKE_FIG_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Reconstruct the true density grid
# ============================================================
w_mix = 0.35
mu1, sigma1 = -2.0, 0.8
mu2, sigma2 = 1.0, 1.3

left = min(mu1 - 6 * sigma1, mu2 - 6 * sigma2)
right = max(mu1 + 6 * sigma1, mu2 + 6 * sigma2)
m_grid = 1000  

xg = np.linspace(left, right, m_grid)

f_true_g = (
    w_mix * norm.pdf(xg, mu1, sigma1)
    + (1.0 - w_mix) * norm.pdf(xg, mu2, sigma2)
)

# ============================================================
# Read saved JSON
# ============================================================
try:
    with open(RESULTS_PATH, "r") as f:
        all_results = json.load(f)
    print(f"Loaded {len(all_results)} saved result entries.")
except FileNotFoundError:
    print(f"Error: Could not find {RESULTS_PATH}. Please update RUN_DIR.")
    exit(1)

# ============================================================
# Plot helpers
# ============================================================
def save_plot(base_path):
    """Helper to save plots using explicit string concatenation to avoid suffix stripping."""
    base_path_str = str(base_path)
    if SAVE_PDF:
        pdf_path = f"{base_path_str}.pdf"
        plt.savefig(pdf_path, dpi=150, bbox_inches="tight")
        print(f"Saved {pdf_path}")
    if SAVE_PNG:
        png_path = f"{base_path_str}.png"
        plt.savefig(png_path, dpi=150, bbox_inches="tight")
        print(f"Saved {png_path}")

def plot_density_from_saved(res):
    """
    Remake density plot using the final saved f_hat_history.
    Handles cases where KDE might not have been saved for a rep.
    """
    if res.get("f_hat_history") is None:
        return

    p = float(res["p_target"])
    alpha_real = float(res["alpha_real"])

    f_hat_history = np.asarray(res["f_hat_history"])
    f_hat_final = f_hat_history[-1]

    plt.figure(figsize=(7, 4))
    plt.plot(xg, f_true_g, linewidth=2, label="True density")
    plt.plot(xg, f_hat_final, linewidth=2, label="Estim. density")
    plt.xlabel("x")
    plt.ylabel("density")
    
    # Force 1 decimal place ONLY on the y-axis
    ax = plt.gca()
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

    plt.grid(True, axis="y", ls="--", alpha=0.3)
    plt.legend()
    plt.tight_layout()

    out_path = REMAKE_FIG_DIR / f"density_saved_p={p:.2f}_alphaReal={alpha_real:.2f}"
    save_plot(out_path)
    plt.close()

def plot_with_trend(M, values, metric_name, p_val, alpha_val, drop_first=20, save_path=None):
    """
    Plots the log-log scaling with trend line, filtering out NaNs and non-positive 
    values (crucial for unbiased sample-based MMD estimates).
    """
    M = np.asarray(M, dtype=float)
    values = np.asarray(values, dtype=float)
    
    # Filter strictly valid values before log
    valid = np.isfinite(M) & np.isfinite(values) & (M > 0) & (values > 0)

    log_M = np.full_like(M, np.nan, dtype=float)
    log_val = np.full_like(values, np.nan, dtype=float)
    log_M[valid] = np.log10(M[valid])
    log_val[valid] = np.log10(values[valid])

    plt.figure(figsize=(6, 4))
    plt.plot(log_M, log_val, "o-", alpha=0.5, markersize=3, label="Distr. Loss")

    fit_mask = valid.copy()
    fit_mask[:drop_first] = False

    if np.count_nonzero(fit_mask) > 5:
        x_fit = log_M[fit_mask]
        y_fit = log_val[fit_mask]
        slope, intercept = np.polyfit(x_fit, y_fit, 1)
        y_pred = slope * x_fit + intercept
        plt.plot(x_fit, y_pred, "r--", linewidth=2, label=f"Fit slope: {slope:.3f}")

    plt.xlabel(r"$\log_{10}$(M$_n$)")

    if metric_name == "W1":
        plt.ylabel(r"$\log_{10}(W_1)$")
    elif metric_name == "MMD":
        plt.ylabel(r"$\log_{10}(\mathrm{MMD})$")
    else:
        # no label 
        pass

    # Force 1 decimal place ONLY on the y-axis
    ax = plt.gca()
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

    plt.legend()
    plt.grid(True, which="both", ls="--", alpha=0.3)
    plt.tight_layout()

    if save_path is not None:
        save_plot(save_path)
        plt.close()
    else:
        plt.show()

def remake_scaling_plots(res):
    p = float(res["p_target"])
    alpha_real = float(res["alpha_real"])
    drop_first = int(res.get("drop_first_for_fit", 20))

    M = np.asarray(res["M"], dtype=float)
    W1 = np.asarray(res["W1"], dtype=float)
    MMD = np.asarray(res.get("MMD", []), dtype=float) 

    run_label = f"p={p:.2f}_alphaReal={alpha_real:.2f}"

    plot_with_trend(
        M,
        W1,
        "W1",
        p,
        alpha_real,
        drop_first=drop_first,
        save_path=REMAKE_FIG_DIR / f"{run_label}_W1",
    )

    if MMD.size > 0:
        plot_with_trend(
            M,
            MMD,
            "MMD",
            p,
            alpha_real,
            drop_first=drop_first,
            save_path=REMAKE_FIG_DIR / f"{run_label}_MMD",
        )

def plot_alpha_vs_mean_slope(metric_key, slopes_key, metric_label, filename, use_error_bars=False):
    unique_ps = sorted(set(float(res["p_target"]) for res in all_results))

    plt.figure(figsize=(6, 4))
    DARK_BLUE = "#244B99"
    RED = "#CC0000"

    for p in unique_ps:
        subset = [r for r in all_results if float(r["p_target"]) == p]

        alpha_plot = np.array([float(r["alpha_real"]) for r in subset])
        means = np.array([float(r[metric_key]) for r in subset])

        order = np.argsort(alpha_plot)
        alpha_plot = alpha_plot[order]
        means = means[order]

        # Use convention where y-axis shows estimated positive exponent magnitude
        vals_plot = -means

        if use_error_bars:
            stds = np.array([
                np.asarray(r[slopes_key], dtype=float).std(ddof=1)
                for r in subset
            ])
            stds = stds[order]

            n_reps = len(subset[0][slopes_key])
            err_plot = 1.96 * stds / np.sqrt(n_reps)

            plt.errorbar(
                alpha_plot,
                vals_plot,
                yerr=err_plot,
                fmt="o-",
                color=DARK_BLUE,
                ecolor=DARK_BLUE,
                elinewidth=1,
                capsize=3,
                label="Empirical",
            )
        else:
            plt.plot(alpha_plot, vals_plot, "o-", color=DARK_BLUE, label="Empirical")

        theo_vals = np.minimum(p, alpha_plot)
        plt.plot(alpha_plot, theo_vals, "--", color=RED, label="Theoretical")

    plt.xlabel(r"$\alpha$")

    if metric_label == "W1":
        plt.ylabel(r"Mean Slope $W_1$")
    elif metric_label == "MMD":
        plt.ylabel(r"Mean Slope $\mathrm{MMD}$")
    else:
        pass

    plt.grid(True, axis="y", ls="--", alpha=0.3)
    plt.ylim(0.0, 0.65)
    plt.yticks(np.arange(0.0, 0.7, 0.1))

    # Force 1 decimal place ONLY on the y-axis
    ax = plt.gca()
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

    handles, labels = plt.gca().get_legend_handles_labels()
    uniq = dict(zip(labels, handles))
    
    # Toggle off legend per user's template to prevent overlapping issues
    legend = False 
    if legend:
        plt.legend(uniq.values(), uniq.keys())

    plt.tight_layout()
    out_path = REMAKE_FIG_DIR / filename
    save_plot(out_path)
    plt.close()

# ============================================================
# Remake all plots
# ============================================================
if __name__ == "__main__":
    for res in all_results:
        remake_scaling_plots(res)
        plot_density_from_saved(res)

    plot_alpha_vs_mean_slope(
        metric_key="mean_slope_W1",
        slopes_key="slopes_W1",
        metric_label="W1",
        filename="alphaReal_vs_mean_slope_W1",
    )

    plot_alpha_vs_mean_slope(
        metric_key="mean_slope_MMD",
        slopes_key="slopes_MMD",
        metric_label="MMD",
        filename="alphaReal_vs_mean_slope_MMD",
    )

### WGAN - Summary Plots

In [ ]:
## just summary plots 

# ============================================================
# Update sizing
# ============================================================
import matplotlib as mpl 
mpl.rcParams["font.sans-serif"] = [
    "Arial",
]

# Smaller font 
mpl.rcParams["font.size"] = 12
mpl.rcParams["axes.titlesize"] = 18
mpl.rcParams["axes.labelsize"] = 18
mpl.rcParams["legend.fontsize"] = 11
mpl.rcParams["legend.title_fontsize"] = 14

# Axis tick label sizes
mpl.rcParams["xtick.labelsize"] = 16
mpl.rcParams["ytick.labelsize"] = 16

# Cleaner editable text in PDF/SVG
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

# ============================================================
# Remake summary plots
# ============================================================
plot_alpha_vs_mean_slope(
    metric_key="mean_slope_W1",
    slopes_key="slopes_W1",
    metric_label="W1",
    filename="alpha_vs_mean_slope_W1",
)

plot_alpha_vs_mean_slope(
    metric_key="mean_slope_MMD",
    slopes_key="slopes_MMD",
    metric_label="MMD",
    filename="alpha_vs_mean_slope_MMD",
)

plot_alpha_vs_mean_slope(
    metric_key="mean_slope_W1",
    slopes_key="slopes_W1",
    metric_label="",
    filename="alpha_vs_mean_slope_W1_noLabel",
)

plot_alpha_vs_mean_slope(
    metric_key="mean_slope_MMD",
    slopes_key="slopes_MMD",
    metric_label="",
    filename="alpha_vs_mean_slope_MMD_noLabel",
)

print(f"\nDone. Remade figures are in:\n{REMAKE_FIG_DIR}")